# 第07章：Softmax — Triton 经典案例

## 前置知识

在学习本章之前，请确保你已经完成以下章节：

- 第01章：Triton 环境搭建与第一个 kernel
- 第02章：Grid、Block 与线程模型
- 第03章：指针运算与内存访问
- 第04章：掩码（Mask）与边界处理
- 第05章：逐元素运算（Element-wise Operations）
- 第06章：归约操作（Reduction）

## 学习目标

通过本章的学习，你将掌握：

1. **Softmax 的数学原理**：理解朴素 Softmax 的溢出问题以及数值稳定版本的推导
2. **Two-Pass Softmax**：用 Triton 实现经典的两遍扫描 Softmax（每个 program 处理一行）
3. **Online Softmax**：理解并实现 Milakov & Gimelshein 提出的在线 Softmax 算法，处理超长行
4. **性能对比**：使用 `triton.testing.perf_report` 对比 Triton 与 PyTorch 原生实现的性能

Softmax 是深度学习中最常见的操作之一（Transformer 中的注意力机制核心），也是 Triton 官方教程中的经典案例。掌握它将为后续学习 FlashAttention 等高级 kernel 打下坚实基础。

## 7.1 Softmax 数学公式

### 朴素公式

给定一个向量 $\mathbf{x} = [x_0, x_1, \dots, x_{N-1}]$，Softmax 的定义为：

$$
\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=0}^{N-1} e^{x_j}}
$$

### 数值溢出问题

当 $x_i$ 很大时（比如 $x_i = 1000$），$e^{x_i}$ 会直接溢出为 `inf`，导致结果变成 `nan`：

```
朴素 Softmax 溢出示例：

  x = [1000, 1001, 999]

  exp(1000) = +inf   ← 溢出！
  exp(1001) = +inf   ← 溢出！
  exp(999)  = +inf   ← 溢出！

  softmax = [inf/inf, inf/inf, inf/inf] = [nan, nan, nan]  ← 全部变成 nan
```

### 数值稳定版本

解决方法：先减去最大值 $M = \max(\mathbf{x})$，利用 Softmax 的平移不变性：

$$
\text{softmax}(x_i) = \frac{e^{x_i - M}}{\sum_{j=0}^{N-1} e^{x_j - M}}
$$

其中 $M = \max(x_0, x_1, \dots, x_{N-1})$

```
数值稳定 Softmax：

  x = [1000, 1001, 999]
  M = max(x) = 1001

  x - M = [-1, 0, -2]

  exp(-1) = 0.368
  exp( 0) = 1.000     ← 最大值变成 0，exp(0) = 1，绝不会溢出
  exp(-2) = 0.135

  sum = 0.368 + 1.000 + 0.135 = 1.503

  softmax = [0.245, 0.665, 0.090]   ← 正确结果，和为 1
```

### 计算步骤总结

```
输入:  x = [x_0, x_1, ..., x_{N-1}]
         │
         ▼
步骤1:  M = max(x_0, x_1, ..., x_{N-1})        ← 求最大值
         │
         ▼
步骤2:  y_i = exp(x_i - M)    对每个元素         ← 减最大值后取 exp
         │
         ▼
步骤3:  S = sum(y_0, y_1, ..., y_{N-1})         ← 求和
         │
         ▼
步骤4:  out_i = y_i / S        对每个元素         ← 归一化
         │
         ▼
输出:  out = [out_0, out_1, ..., out_{N-1}]
```

这就是经典的 **Two-Pass（两遍扫描）** 方法：第一遍求最大值，第二遍计算 exp、求和、归一化。

In [ ]:
import torch
import triton
import triton.language as tl

## 7.2 Two-Pass Softmax：每个 Program 处理一行

### 核心思路

对于一个 `(M, N)` 的矩阵，我们对每一行独立地做 Softmax。最简单的方案是：

- 启动 `M` 个 program（每行一个）
- 每个 program 把整行数据加载到 SRAM 中
- 在 SRAM 中完成 max → exp → sum → divide 全部计算

```
矩阵布局 (M x N)：

         列 0    列 1    列 2   ...   列 N-1
        ┌───────┬───────┬───────┬───┬─────────┐
 行 0   │ x_0,0 │ x_0,1 │ x_0,2 │...│ x_0,N-1 │  ← program 0 处理
        ├───────┼───────┼───────┼───┼─────────┤
 行 1   │ x_1,0 │ x_1,1 │ x_1,2 │...│ x_1,N-1 │  ← program 1 处理
        ├───────┼───────┼───────┼───┼─────────┤
  ...   │  ...  │  ...  │  ...  │...│   ...   │
        ├───────┼───────┼───────┼───┼─────────┤
 行 M-1 │x_M-1,0│x_M-1,1│x_M-1,2│...│x_M-1,N-1│  ← program M-1 处理
        └───────┴───────┴───────┴───┴─────────┘

每个 program 的工作流程：

  ┌──────────────────────────────────────┐
  │  1. 加载整行到 SRAM                    │
  │  2. row_max = tl.max(row)             │
  │  3. row = exp(row - row_max)          │
  │  4. row_sum = tl.sum(row)             │
  │  5. row = row / row_sum               │
  │  6. 写回结果                           │
  └──────────────────────────────────────┘
```

**限制**：`BLOCK_SIZE` 必须大于等于行宽 `N`，也就是说整行必须能放进一个 block。这适用于 `N` 不太大的情况（比如 vocabulary size 较小时）。

In [ ]:
@triton.jit
def softmax_kernel(
    input_ptr,       # 输入矩阵指针，形状 (M, N)
    output_ptr,      # 输出矩阵指针，形状 (M, N)
    n_cols,          # 列数 N
    input_row_stride,   # 输入矩阵的行步长
    output_row_stride,  # 输出矩阵的行步长
    BLOCK_SIZE: tl.constexpr,  # 每个 block 处理的列数，必须 >= n_cols
):
    # ---- 确定当前 program 处理哪一行 ----
    row_idx = tl.program_id(0)

    # 计算当前行的起始指针
    row_start_ptr = input_ptr + row_idx * input_row_stride

    # 生成列偏移 [0, 1, 2, ..., BLOCK_SIZE-1]
    col_offsets = tl.arange(0, BLOCK_SIZE)

    # 构造指针 + 掩码（处理 N 不是 BLOCK_SIZE 整数倍的情况）
    input_ptrs = row_start_ptr + col_offsets
    mask = col_offsets < n_cols

    # ---- 第一步：加载整行数据 ----
    # 对于超出范围的位置，用 -inf 填充（不影响 max 和 exp 计算）
    row = tl.load(input_ptrs, mask=mask, other=float('-inf'))

    # ---- 第二步：数值稳定 Softmax ----
    # 2a. 求最大值
    row_max = tl.max(row, axis=0)

    # 2b. 减去最大值后取 exp
    numerator = tl.exp(row - row_max)

    # 2c. 求和
    denominator = tl.sum(numerator, axis=0)

    # 2d. 归一化
    softmax_output = numerator / denominator

    # ---- 第三步：写回结果 ----
    output_row_start_ptr = output_ptr + row_idx * output_row_stride
    output_ptrs = output_row_start_ptr + col_offsets
    tl.store(output_ptrs, softmax_output, mask=mask)

In [ ]:
def softmax(x: torch.Tensor) -> torch.Tensor:
    """使用 Triton kernel 计算 Softmax（对最后一个维度）。"""
    assert x.ndim == 2, "输入必须是 2D 张量 (M, N)"
    M, N = x.shape

    # 分配输出张量
    output = torch.empty_like(x)

    # BLOCK_SIZE 必须是 2 的幂且 >= N
    BLOCK_SIZE = triton.next_power_of_2(N)

    # 启动 M 个 program，每个处理一行
    grid = (M,)

    softmax_kernel[grid](
        x, output,
        N,
        x.stride(0),
        output.stride(0),
        BLOCK_SIZE=BLOCK_SIZE,
    )

    return output


# ---- 正确性测试 ----
torch.manual_seed(42)

# 测试多种形状
test_shapes = [(128, 64), (64, 128), (256, 781), (1, 1024), (512, 100)]

for M, N in test_shapes:
    x = torch.randn(M, N, device='cuda', dtype=torch.float32)

    # Triton 版本
    y_triton = softmax(x)

    # PyTorch 参考版本
    y_torch = torch.softmax(x, dim=-1)

    # 比较
    max_diff = (y_triton - y_torch).abs().max().item()
    assert torch.allclose(y_triton, y_torch, atol=1e-5), f"形状 ({M}, {N}) 不匹配！最大差异: {max_diff}"
    print(f"形状 ({M:>4}, {N:>4}) ✓  最大差异: {max_diff:.2e}")

print("\n所有测试通过！Triton Softmax 与 PyTorch 结果一致。")

## 7.3 Online Softmax（在线 Softmax）

### 为什么需要 Online Softmax？

上面的 Two-Pass 方法要求 `BLOCK_SIZE >= N`，即整行必须一次性加载到 SRAM 中。
当行非常长时（例如 N = 32768 或更大），这可能超出 SRAM 容量。

**Online Softmax** 由 Milakov & Gimelshein (2018) 提出，核心思想是：
将一行分成多个 block，**逐块扫描**，用递推公式在线更新 `max` 和 `sum`。

### 算法推导

假设我们将一行分为多个 block，依次处理 block 0, 1, 2, ...：

```
一行数据:  [  block_0  |  block_1  |  block_2  |  ...  |  block_K-1  ]
              ↓            ↓            ↓                    ↓
           处理第1块     处理第2块     处理第3块     ...    处理最后1块
```

#### 递推公式（前向扫描：求 max 和 sum）

初始化：
- $m_0 = -\infty$ （running max）
- $l_0 = 0$（running sum of exp）

处理第 $k$ 个 block（记该 block 的数据为 $\mathbf{b}_k$）：

```
1. block_max = max(b_k)                          ← 当前 block 的最大值

2. m_new = max(m_old, block_max)                  ← 更新全局最大值

3. correction = exp(m_old - m_new)                ← 修正因子
   （如果 m_new > m_old，之前累积的 sum 需要乘以修正因子缩小）
   （如果 m_new == m_old，correction = 1，无需修正）

4. l_new = l_old * correction + sum(exp(b_k - m_new))  ← 更新累积和
```

#### 直觉理解

```
假设已经处理了 block_0 和 block_1，当前状态：
  m_old = 5.0    (目前见过的最大值)
  l_old = 12.3   (基于 m_old=5.0 计算的 exp 之和)

现在处理 block_2 = [3.0, 7.0, 2.0]：
  block_max = 7.0
  m_new = max(5.0, 7.0) = 7.0   ← 最大值更新了！

  问题：之前的 l_old = 12.3 是基于 exp(x - 5.0) 算的
        现在最大值变成了 7.0，需要改成 exp(x - 7.0)

  修正：exp(x - 7.0) = exp(x - 5.0) * exp(5.0 - 7.0)
                      = exp(x - 5.0) * exp(-2.0)

  correction = exp(5.0 - 7.0) = exp(-2.0) ≈ 0.135

  l_new = 12.3 * 0.135 + exp(3.0-7.0) + exp(7.0-7.0) + exp(2.0-7.0)
        = 1.661 + 0.018 + 1.000 + 0.007
        = 2.686
```

#### 第二遍：归一化

扫描完所有 block 后得到最终的 $M_{\text{final}}$ 和 $L_{\text{final}}$，再做第二遍：

$$
\text{output}_i = \frac{e^{x_i - M_{\text{final}}}}{L_{\text{final}}}
$$

```
Online Softmax 完整流程：

  ┌─────────────────────────────────────────────────┐
  │ 第一遍（前向）：逐块求 running_max 和 running_sum  │
  │                                                 │
  │   for k in range(K):                            │
  │     加载 block_k                                 │
  │     更新 m_new, l_new（带 correction）            │
  │                                                 │
  │   得到 M_final, L_final                          │
  └──────────────────────┬──────────────────────────┘
                         │
                         ▼
  ┌─────────────────────────────────────────────────┐
  │ 第二遍（归一化）：逐块写回结果                      │
  │                                                 │
  │   for k in range(K):                            │
  │     加载 block_k                                 │
  │     output = exp(block_k - M_final) / L_final   │
  │     写回 output                                  │
  └─────────────────────────────────────────────────┘
```

这样每个 block 只需要 `BLOCK_SIZE` 大小的 SRAM，可以处理任意长度的行！

In [ ]:
@triton.jit
def online_softmax_kernel(
    input_ptr,
    output_ptr,
    n_cols,
    input_row_stride,
    output_row_stride,
    BLOCK_SIZE: tl.constexpr,  # 每次处理的列数（不必 >= n_cols）
):
    # ---- 确定当前 program 处理哪一行 ----
    row_idx = tl.program_id(0)
    row_start_ptr = input_ptr + row_idx * input_row_stride

    # ====================================================
    # 第一遍：逐块扫描，求 running_max 和 running_sum
    # ====================================================
    running_max = float('-inf')  # m_old
    running_sum = 0.0            # l_old

    # 计算需要多少个 block 才能覆盖整行
    # 用 tl.cdiv 做向上取整除法
    num_blocks = tl.cdiv(n_cols, BLOCK_SIZE)

    for block_idx in range(0, num_blocks):
        # 当前 block 的列偏移
        col_offsets = block_idx * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = col_offsets < n_cols

        # 加载当前 block
        block_data = tl.load(row_start_ptr + col_offsets, mask=mask, other=float('-inf'))

        # 当前 block 的最大值
        block_max = tl.max(block_data, axis=0)

        # 更新 running_max
        new_max = tl.maximum(running_max, block_max)

        # 修正因子：之前的 sum 需要根据新的 max 调整
        correction = tl.exp(running_max - new_max)

        # 当前 block 在新 max 下的 exp 之和
        block_exp_sum = tl.sum(tl.exp(block_data - new_max), axis=0)

        # 更新 running_sum
        running_sum = running_sum * correction + block_exp_sum

        # 更新 running_max
        running_max = new_max

    # 此时 running_max = M_final, running_sum = L_final

    # ====================================================
    # 第二遍：逐块归一化并写回
    # ====================================================
    output_row_start_ptr = output_ptr + row_idx * output_row_stride

    for block_idx in range(0, num_blocks):
        col_offsets = block_idx * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = col_offsets < n_cols

        # 重新加载当前 block
        block_data = tl.load(row_start_ptr + col_offsets, mask=mask, other=float('-inf'))

        # 计算 softmax 输出
        block_output = tl.exp(block_data - running_max) / running_sum

        # 写回结果
        tl.store(output_row_start_ptr + col_offsets, block_output, mask=mask)

In [ ]:
def online_softmax(x: torch.Tensor, block_size: int = 256) -> torch.Tensor:
    """使用 Online Softmax kernel，可以处理任意长度的行。

    Args:
        x: 输入张量，形状 (M, N)
        block_size: 每次处理的列数，不必 >= N
    """
    assert x.ndim == 2, "输入必须是 2D 张量 (M, N)"
    M, N = x.shape

    output = torch.empty_like(x)

    # block_size 取 2 的幂
    BLOCK_SIZE = triton.next_power_of_2(block_size)

    grid = (M,)

    online_softmax_kernel[grid](
        x, output,
        N,
        x.stride(0),
        output.stride(0),
        BLOCK_SIZE=BLOCK_SIZE,
    )

    return output


# ---- 正确性测试（包含大 N 场景）----
torch.manual_seed(42)

test_shapes = [
    (128, 64),     # 小矩阵
    (64, 512),     # 中等
    (32, 2048),    # 较大
    (16, 8192),    # 大 N：一个 block 装不下，必须多次循环
    (8, 16384),    # 更大的 N
]

# 使用较小的 block_size（256），确保 N > block_size 时也能正确工作
for M, N in test_shapes:
    x = torch.randn(M, N, device='cuda', dtype=torch.float32)

    y_online = online_softmax(x, block_size=256)
    y_torch = torch.softmax(x, dim=-1)

    max_diff = (y_online - y_torch).abs().max().item()
    assert torch.allclose(y_online, y_torch, atol=1e-4), (
        f"形状 ({M}, {N}) 不匹配！最大差异: {max_diff}"
    )
    num_blocks = (N + 255) // 256
    print(f"形状 ({M:>4}, {N:>5}) | blocks/row: {num_blocks:>2} | 最大差异: {max_diff:.2e} ✓")

print("\n所有 Online Softmax 测试通过！")

## 7.4 性能对比

接下来我们使用 Triton 内置的 `triton.testing.perf_report` 工具来对比：

1. **Triton Softmax**（Two-Pass，单 block 版本）
2. **PyTorch `torch.softmax`**

我们固定行数 `M = 4096`，逐步增大列数 `N`，观察吞吐量（GB/s）随 `N` 的变化趋势。

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['N'],  # x 轴：列数
        x_vals=[128 * i for i in range(2, 33)],  # N 从 256 到 4096
        line_arg='provider',
        line_vals=['triton', 'torch'],
        line_names=['Triton Softmax', 'PyTorch Softmax'],
        styles=[('blue', '-'), ('green', '-')],
        ylabel='GB/s',
        plot_name='softmax-performance',
        args={'M': 4096},  # 固定行数
    )
)
def benchmark(M, N, provider):
    x = torch.randn(M, N, device='cuda', dtype=torch.float32)
    quantiles = [0.5, 0.2, 0.8]  # 中位数、20%、80%

    if provider == 'triton':
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: softmax(x), quantiles=quantiles
        )
    elif provider == 'torch':
        ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: torch.softmax(x, dim=-1), quantiles=quantiles
        )

    # 计算吞吐量（GB/s）
    # 读入 M*N 个 float32 + 写出 M*N 个 float32 = 2 * M * N * 4 bytes
    gbps = lambda ms: 2 * M * N * x.element_size() * 1e-9 / (ms * 1e-3)
    return gbps(ms), gbps(max_ms), gbps(min_ms)


# 运行 benchmark（结果会以表格或图表形式展示）
benchmark.run(show_plots=True, print_data=True)

## 7.5 总结

### 本章核心知识点

| 知识点 | 说明 |
|--------|------|
| 数值稳定 Softmax | 先减最大值再取 exp，避免溢出 |
| Two-Pass 方法 | 第一遍求 max，第二遍 exp → sum → divide；要求 BLOCK_SIZE >= N |
| `tl.max` / `tl.sum` | Triton 内置的归约操作，在 SRAM 中高效执行 |
| Online Softmax | 逐块扫描 + 修正因子 correction = exp(m_old - m_new)，无需整行装入 SRAM |
| 递推公式 | m_new = max(m_old, block_max)；l_new = l_old * correction + block_exp_sum |
| `triton.testing.perf_report` | Triton 官方 benchmark 工具，支持多曲线对比和百分位统计 |

### 练习

1. **Log-Softmax**：实现 `log_softmax(x_i) = x_i - max(x) - log(sum(exp(x - max(x))))`。
   提示：不需要先算 softmax 再取 log，可以直接用公式避免精度损失。

2. **Softmax Backward**：实现 Softmax 的反向传播 kernel。
   给定前向输出 $y$ 和上游梯度 $dy$，反向公式为：
   $dx_i = y_i \cdot (dy_i - \sum_j y_j \cdot dy_j)$

3. **Temperature Scaling**：修改 Softmax kernel，加入温度参数 $T$：
   $\text{softmax}(x_i; T) = \frac{e^{x_i / T}}{\sum_j e^{x_j / T}}$
   观察不同温度下输出分布的变化（T→0 趋近 argmax，T→∞ 趋近均匀分布）。